In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

# Parameter
img_height, img_width = 224, 224
model_path = '/content/drive/MyDrive/BlockchainAI/panadol_v4_clean.h5'
demo_dir = '/content/drive/MyDrive/BlockchainAI/demo_test'

print("✅ Setup selesai.")

In [ ]:
def find_medicine_package(image):
    h, w = image.shape[:2]
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    min_area = (h * w) * 0.02

    # Strategi 1: Canny
    edges = cv2.Canny(blurred, 20, 120)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    dilated = cv2.dilate(edges, kernel, iterations=4)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_rect = None
    max_area = 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > max_area and area > min_area:
            max_area = area
            best_rect = cv2.boundingRect(cnt)

    if best_rect is not None:
        x, y, bw, bh = best_rect
        # Jika bounding box mencakup >85% gambar, anggap tidak valid
        if (bw * bh) < (h * w) * 0.85:
            return best_rect

    # Strategi 2: Adaptive threshold
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 15, 3)
    kernel2 = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel2)
    contours2, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_rect2 = None
    max_area2 = 0
    for cnt in contours2:
        area = cv2.contourArea(cnt)
        if area > max_area2 and area > min_area:
            max_area2 = area
            best_rect2 = cv2.boundingRect(cnt)

    if best_rect2 is not None:
        x, y, bw, bh = best_rect2
        if (bw * bh) < (h * w) * 0.85:
            return best_rect2

    # Strategi 3: Otsu
    _, otsu = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel3 = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    otsu = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, kernel3)
    contours3, _ = cv2.findContours(otsu, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_rect3 = None
    max_area3 = 0
    for cnt in contours3:
        area = cv2.contourArea(cnt)
        if area > max_area3 and area > min_area:
            max_area3 = area
            best_rect3 = cv2.boundingRect(cnt)

    if best_rect3 is not None:
        x, y, bw, bh = best_rect3
        if (bw * bh) < (h * w) * 0.85:
            return best_rect3

    # Tidak ada kontur valid → kemungkinan non‑obat
    return None

In [ ]:
def crop_and_enhance(image, rect):
    x, y, w_rect, h_rect = rect

    pad_x = int(w_rect * 0.02)
    pad_y = int(h_rect * 0.02)
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(image.shape[1], x + w_rect + pad_x)
    y2 = min(image.shape[0], y + h_rect + pad_y)

    cropped = image[y1:y2, x1:x2]
    if cropped.size == 0:
        cropped = image

    enhanced = cv2.resize(cropped, (img_height, img_width))

    kernel_sharpen = np.array([[-1, -1, -1],
                                [-1,  9, -1],
                                [-1, -1, -1]])
    enhanced = cv2.filter2D(enhanced, -1, kernel_sharpen)

    lab = cv2.cvtColor(enhanced, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.merge([l, a, b])
    enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

    return enhanced

In [ ]:
def is_medicine_package(image, strict=False):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # 1. Area putih / krem
    mask_white = cv2.inRange(hsv, (0, 0, 180), (180, 50, 255))
    white_ratio = np.sum(mask_white > 0) / mask_white.size

    if not strict:
        return white_ratio >= 0.02

    # 2. Blob teks/logo
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 11, 2)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    text_blobs = sum(1 for cnt in contours if 50 < cv2.contourArea(cnt) < 5000)

    # 3. Filter warna alam
    mask_blue = cv2.inRange(hsv, (90, 50, 50), (130, 255, 255))
    mask_green = cv2.inRange(hsv, (35, 50, 50), (85, 255, 255))
    nature_ratio = (np.sum(mask_blue > 0) + np.sum(mask_green > 0)) / mask_blue.size

    # 4. Filter warna mencolok (merah, kuning, biru terang)
    mask_red = cv2.inRange(hsv, (0, 100, 100), (10, 255, 255))
    mask_yellow = cv2.inRange(hsv, (20, 100, 100), (35, 255, 255))
    mask_bright_blue = cv2.inRange(hsv, (100, 100, 100), (120, 255, 255))
    bright_ratio = (np.sum(mask_red > 0) + np.sum(mask_yellow > 0) + np.sum(mask_bright_blue > 0)) / mask_red.size

    has_white = white_ratio >= 0.03
    has_text = text_blobs >= 3
    not_nature = nature_ratio < 0.4
    not_bright = bright_ratio < 0.1

    return has_white and has_text and not_nature and not_bright

In [ ]:
# ========================================================
# BAGIAN 4: Load Model & Fungsi Prediksi
# ========================================================
# Load model v4
model = tf.keras.models.load_model(model_path)
verify_model = model
print("✅ Model v4 dimuat")

def predict_medicine(image_path, threshold=0.5, confidence_floor=0.80, white_ratio_min=0.03, min_text_blobs=3):
    img = cv2.imread(image_path)
    if img is None:
        return "ERROR", "Gambar tidak ditemukan", 0.0

    rect = find_medicine_package(img)
    if rect is None:
        rect = (0, 0, img.shape[1], img.shape[0])

    enhanced = crop_and_enhance(img, rect)
    if enhanced is None:
        return "ERROR", "Gagal memproses gambar", 0.0

    # Prediksi model
    img_rgb = cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)
    img_array = np.expand_dims(img_rgb, axis=0) / 255.0
    pred = model.predict(img_array, verbose=0)[0][0]

    if pred >= threshold:
        label = "❌ PALSU"
        confidence = pred
    else:
        label = "✅ ASLI"
        confidence = 1.0 - pred

    # 1. Confidence floor
    if confidence < confidence_floor:
        return "DITOLAK", "❌ Objek tidak dikenal / bukan obat", confidence

    # 2. Ciri kemasan: area putih/krem
    hsv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV)
    mask_white = cv2.inRange(hsv, (0, 0, 180), (180, 50, 255))
    white_ratio = np.sum(mask_white > 0) / mask_white.size

    # 3. Ciri kemasan: kontur kecil (teks/logo)
    gray = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    text_blobs = sum(1 for cnt in contours if 50 < cv2.contourArea(cnt) < 5000)

    has_white = white_ratio >= white_ratio_min
    has_text = text_blobs >= min_text_blobs

    if not (has_white or has_text):
        return "DITOLAK", "❌ Objek bukan obat / tidak ada ciri kemasan", confidence

    return "TERVERIFIKASI", label, confidence

In [ ]:
# ========================================================
# BAGIAN 5: Batch Testing di demo_test
# ========================================================
print("="*70)
print(f"BATCH TESTING PADA FOLDER: {demo_dir}")
print("="*70)

if not os.path.exists(demo_dir):
    print("❌ Folder demo_test tidak ditemukan.")
else:
    test_files = sorted([f for f in os.listdir(demo_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if not test_files:
        print("⚠️ Folder demo_test kosong.")
    else:
        results = {"ASLI": 0, "PALSU": 0, "DITOLAK": 0, "ERROR": 0}
        for fname in test_files:
            path = os.path.join(demo_dir, fname)
            status, label, confidence = predict_medicine(path, threshold=0.5)
            if status == "DITOLAK":
                results["DITOLAK"] += 1
                print(f"{fname:<40} {label}")
            elif status == "ERROR":
                results["ERROR"] += 1
                print(f"{fname:<40} {label}")
            else:
                symbol = "✅" if label == "✅ ASLI" else "❌"
                print(f"{fname:<40} {symbol} {label}  (confidence: {confidence*100:.1f}%)")
                if label == "✅ ASLI":
                    results["ASLI"] += 1
                else:
                    results["PALSU"] += 1

        print("\n" + "="*70)
        print("RINGKASAN")
        print("="*70)
        print(f"✅ ASLI    : {results['ASLI']}")
        print(f"❌ PALSU   : {results['PALSU']}")
        print(f"🚫 DITOLAK : {results['DITOLAK']} (bukan obat / tidak terdeteksi)")
        print(f"⚠️  ERROR  : {results['ERROR']}")

In [ ]:
def visualize_prediction(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print("Gambar tidak ditemukan.")
        return
    rect = find_medicine_package(img)
    if rect:
        x, y, w, h = rect
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 3)
        enhanced = crop_and_enhance(cv2.imread(image_path), rect)
    else:
        enhanced = None

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Deteksi")
    plt.axis('off')
    if enhanced is not None:
        plt.subplot(1, 2, 2)
        plt.imshow(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB))
        plt.title("Crop & Enhance")
        plt.axis('off')
    plt.show()

# Contoh penggunaan
visualize_prediction('/content/drive/MyDrive/BlockchainAI/demo_test/c1aca82d-af01-4898-90df-9ac15bb137cd.jpg')